In [1]:
 pip install torch numpy


  Using cached torch-2.12.1-cp313-cp313-win_amd64.whl.metadata (31 kB)
   ---------------------------------------- 0.0/123.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/123.0 MB ? eta -:--:--
   ---------------------------------------- 0.5/123.0 MB 2.3 MB/s eta 0:00:54
   ---------------------------------------- 0.8/123.0 MB 2.3 MB/s eta 0:00:53
   ---------------------------------------- 1.3/123.0 MB 2.0 MB/s eta 0:01:02
    --------------------------------------- 1.8/123.0 MB 2.0 MB/s eta 0:01:02
    --------------------------------------- 2.1/123.0 MB 2.0 MB/s eta 0:01:00
    --------------------------------------- 2.6/123.0 MB 2.1 MB/s eta 0:00:59
    --------------------------------------- 2.9/123.0 MB 1.9 MB/s eta 0:01:02
   - -------------------------------------- 3.4/123.0 MB 2.0 MB/s eta 0:01:01
   - -------------------------------------- 3.9/123.0 MB 2.0 MB/s eta 0:01:01
   - -------------------------------------- 4.5/123.0 MB 2.0 MB/s eta 0:00:59
   - -

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import random
from collections import deque

In [2]:
class DQNetwork(nn.Module):
    def __init__(self, state_size, action_size):
        super(DQNetwork, self).__init__()

        self.fc1 = nn.Linear(state_size, 64)
        self.fc2 = nn.Linear(64, 64)
        self.fc3 = nn.Linear(64, action_size)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        return self.fc3(x)

In [3]:
class DQNAgent:
    def __init__(self, state_size, action_size):
        self.state_size = state_size
        self.action_size = action_size

        self.memory = deque(maxlen=5000)

        # Hyperparameters
        self.gamma = 0.95
        self.epsilon = 1.0
        self.epsilon_min = 0.01
        self.epsilon_decay = 0.995
        self.learning_rate = 0.001
        # Models
        self.model = DQNetwork(state_size, action_size)
        self.optimizer = optim.Adam(self.model.parameters(), lr=self.learning_rate)
        self.criterion = nn.MSELoss()

In [4]:
def remember(self, state, action, reward, next_state, done):
        self.memory.append((state, action, reward, next_state, done))

In [5]:
def act(self, state):
        if np.random.rand() <= self.epsilon:
            return random.randrange(self.action_size)

        state = torch.FloatTensor(state)
        q_values = self.model(state)
        return torch.argmax(q_values).item()

In [10]:
class DQNAgent:
    def __init__(self, state_size, action_size):
        self.state_size = state_size
        self.action_size = action_size

        self.memory = deque(maxlen=5000)

        # Hyperparameters
        self.gamma = 0.95
        self.epsilon = 1.0
        self.epsilon_min = 0.01
        self.epsilon_decay = 0.995
        self.learning_rate = 0.001
        # Models
        self.model = DQNetwork(state_size, action_size)
        self.optimizer = optim.Adam(self.model.parameters(), lr=self.learning_rate)
        self.criterion = nn.MSELoss()

    def remember(self, state, action, reward, next_state, done):
        # Ensure state, next_state are stored as numpy arrays without the extra batch dim
        # state is coming in as [1, state_size], so squeeze it to [state_size] for memory storage
        self.memory.append((state.squeeze(0), action, reward, next_state.squeeze(0), done))

    def act(self, state):
        if np.random.rand() <= self.epsilon:
            return random.randrange(self.action_size)
        # Predict action from Q-network
        # state comes in as [1, state_size]. Convert to tensor as [1, state_size]
        state_tensor = torch.FloatTensor(state)
        with torch.no_grad():
            q_values = self.model(state_tensor)
        return torch.argmax(q_values).item()

    def replay(self, batch_size):
        if len(self.memory) < batch_size:
            return

        minibatch = random.sample(self.memory, batch_size)

        # Each e[0] and e[3] will now be [state_size] numpy arrays because of the change in remember
        # Concatenate them to form [batch_size, state_size] tensors
        states = torch.FloatTensor(np.array([e[0] for e in minibatch]))
        actions = torch.LongTensor([e[1] for e in minibatch])
        rewards = torch.FloatTensor([e[2] for e in minibatch])
        next_states = torch.FloatTensor(np.array([e[3] for e in minibatch]))
        dones = torch.FloatTensor([e[4] for e in minibatch])

        # Get current Q values from model. self.model(states) will now output [batch_size, action_size]
        current_q_values = self.model(states).gather(1, actions.unsqueeze(1)).squeeze(1)

        # Get next Q values from model (used as target model in vanilla DQN for stability)
        with torch.no_grad():
            next_q_values = self.model(next_states).max(1)[0]

        # Compute the expected Q values
        expected_q_values = rewards + self.gamma * next_q_values * (1 - dones)

        # Compute loss
        loss = self.criterion(current_q_values, expected_q_values)

        # Optimize the model
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

        if self.epsilon > self.epsilon_min:
            self.epsilon *= self.epsilon_decay